# サイモンのアルゴリズム(概要)

サイモンのアルゴリズムについて説明します。

このアルゴリズムでは、$n$ビットの入力と出力を持つ関数 $f_s(x)$($s$ は任意の$n$ビット列)について、以下のいずれかが成り立つと仮定します。

ケース1: 異なる入力に対して常に異なる出力を返す(1対1対応)

ケース2: 入力 $x, x'$ について、$x' = x\oplus s$ ならば $f_s(x) = f_s(x')$ が成り立つ。つまり、2つの入力に対して同じ出力を返す。

このアルゴリズムは、オラクルが上記のケース1かケース2かを判定します。

具体的な量子回路は以下の通りです。$U_f$ の内容は、上記のケース2かつ $s=1001$ の場合について示しています。
量子ビット数は $2n$ です。

<img src="img/103_img.png" width="50%">

状態を確認しましょう。

$$
\begin{align}
\lvert \psi_1\rangle &= \frac{1}{\sqrt{2^n}} \biggl(\otimes^n H\lvert 0\rangle \biggr) \lvert 0\rangle^{\otimes n} \\
&= \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n-1} \lvert x\rangle \lvert 0\rangle^{\otimes n}
\end{align}
$$

次に、$\lvert \psi_2 \rangle$ を考えます。

ここで、$f_s(x)$ に対して以下のオラクルゲート $U_f$ を用意します。

$$
U_f \lvert x \rangle \lvert 0 \rangle = \lvert x \rangle \lvert f_s(x) \rangle
$$

この $U_f$ を使うと、

$$
\lvert \psi_2 \rangle = \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n-1} \lvert x\rangle \lvert f_s(x)\rangle
$$

したがって、$\lvert \psi_3 \rangle$ は以下のようになります。

$$
\lvert \psi_3 \rangle = \frac{1}{2^n} \sum_{x=0}^{2^n-1}\sum_{y=0}^{2^n-1} (-1)^{x\cdot y} \lvert y\rangle \lvert f_s(x)\rangle
$$

では、$f_s(x)$ が以下のような場合、$\lvert y \rangle$ の測定結果がどうなるかを考えます。

ケース1: 異なる入力に対して常に異なる出力を返す(1対1対応)

すべての測定結果が等しい確率で得られます。

ケース2: 入力 $x, x'$ について、$x' = x\oplus s$ ならば $f_s(x) = f_s(x')$ が成り立つ。つまり、2つの入力に対して同じ出力を返す。

状態 $\lvert y \rangle \lvert f_s(x) \rangle = \lvert y \rangle \lvert f_s(x\oplus s) \rangle$ の振幅 $A(y, x)$ に注目します。

$$
A(y, x) = \frac{1}{2^n} \{(-1)^{x\cdot y} + (-1)^{(x\oplus s) \cdot y}\}
$$

式から分かるように、$y\cdot s \equiv 1 \bmod2$ となる $y$ の振幅は打ち消し合って $0$ になります。
したがって、$y\cdot s \equiv 0 \bmod2$ となる $y$ だけが測定されます。

ケース1、ケース2いずれの場合も、そのような異なる $y$ が$n$個("00...0"を除く)測定によって得られれば、それらすべての $y$ について $y\cdot s' \equiv 0 \bmod2$ となる $s'$ を決定できます。

ケース1では、$s'$ は完全にランダムです。
一方、ケース2では、$s' = 0\oplus s'$ であるため、常に $f_s(s') = f_s(0)$ が成り立ちます。

したがって、ケース1から確率 $1 / 2^n$ で $f_s(s') = f_s(0)$ となる $s'$ が得られてしまう場合を除けば、オラクルゲートを使って $s'$ がケース1とケース2のどちらから得られたものかを確認できます。
以上から、オラクルを判定できます。

最後に、オラクルゲート $U_f$ の実装を考えます。

ケース1では、出力が入力 $x$ と1対1対応していれば十分です。
簡単のため、ランダムに $X$ ゲートを挿入する回路を考えます。

ケース2は少し複雑です。
まず、$CX$ ゲートによって以下の状態を作ります。

$$
\lvert \psi_{1a} \rangle = \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n-1} \lvert x\rangle \lvert x\rangle
$$

次に、$s_i=1$ となる最小のインデックス $i'$ について、$x_{i'} = 0$ の場合のみ補助レジスタと $s$ のXORを取ります。
その結果、以下の $\lvert\psi_2\rangle$ が得られます。

$$
\begin{align}
\lvert \psi_{2} \rangle &= \frac{1}{\sqrt{2^n}} \biggl(\sum_{\{x_{i'}=0\}} \lvert x\rangle \lvert x \oplus s\rangle + \sum_{\{x_{i'}=1\}} \lvert x\rangle \lvert x\rangle \biggr) \\
&= \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n-1} \lvert x\rangle \lvert f_s(x)\rangle
\end{align}
$$

計算により、この $f_s(x)$ がケース2を満たすことを確認できます。

Blueqatでこれを実装してみましょう。

In [1]:
from blueqat import Circuit
import numpy as np
from collections import Counter as _Counter

# Compatibility note: the new blueqat SDK reports measurement bitstrings with
# qubit 0 as the right-most character (standard binary ordering), while this
# notebook (like the old blueqat 0.3.x) reads qubit 0 as the left-most
# character. Patch Circuit.run once so sampled bitstrings keep matching the
# qubit ordering used throughout this notebook's explanation.
if not getattr(Circuit, '_qubit0_leftmost_patched', False):
    _original_run = Circuit.run
    def _run_qubit0_leftmost(self, *args, **kwargs):
        result = _original_run(self, *args, **kwargs)
        if isinstance(result, _Counter):
            return _Counter({key[::-1]: count for key, count in result.items()})
        return result
    Circuit.run = _run_qubit0_leftmost
    Circuit._qubit0_leftmost_patched = True

2種類のオラクルゲート $U_f$ を用意する関数を作ります。

In [2]:
def oracle_1(c, s):
    _n = len(s)
    for i in range(_n):
        if np.random.rand() > 0.5:
            c.x[i]
    for i in range(_n):
        c.cx[i, i + _n]
        
def oracle_2(c, s):
    _n = len(s)
    flag = 0
    for i, si in enumerate(reversed(s)):
        c.cx[i, i + _n]
        if si == '1' and flag == 0:
            c.x[i]
            for j, sj in enumerate(s):
                if sj == '1':
                    c.cx[i, j + _n]
            c.x[i]
            flag = 1

以下がアルゴリズムの本体です。
まず、乱数を使ってオラクル(2種類のうちどちらか)と、求めたい $s$ を決定します。
(以下では、上記の図に示した量子回路を再現するために値を固定しています。)

In [3]:
n = 4
N = np.random.randint(1, 2**n-1)
s = bin(N)[2:].zfill(n)
which_oracle = np.random.rand()

### to reproduce the quantum circuit shown in the figure above ###
### Erasing these two lines will randomly determine s and oracle###
s = "1001" 
which_oracle = 0
######

c = Circuit(n * 2)
c.h[:n]

if which_oracle > 0.5:
    oracle_1(c, s)
    oracle = "oracle 1"
else:
    oracle_2(c, s)
    oracle = "oracle 2"
    
c.h[:n].m[:n]
res = c.run(shots = 1000)

In [4]:
res

Counter({'11110000': 144,
         '01100000': 132,
         '00000000': 131,
         '01000000': 124,
         '10010000': 121,
         '10110000': 120,
         '00100000': 116,
         '11010000': 112})

サンプリング結果から、'00...0' 以外の結果を$n$個抽出します。

In [5]:
res_list = list(res.keys())
_res_list = []
for i in res_list:
    if i[:n] != '0'*4:
        _res_list.append(i[:n])
    if len(_res_list) == 4:
        break
            
print(_res_list)

['1101', '1111', '1001', '0110']


抽出した結果から $s'$ を求めます。
(ここでは単純にブルートフォースで条件に合う $s'$ を探していますが、線形代数を使えば効率的に求めることができます。)

ケース2のオラクルが選ばれていれば、得られる $s'$ は $s$ と一致するはずです。

In [6]:
for i in range(2**n):
    l = bin(i)[2:].zfill(n)
    flag = 1
    for sampled in _res_list: 
        mod = np.sum(np.array(list(l), dtype = np.int64) * np.array(list(sampled), dtype = np.int64)) % 2
        if mod:
            flag = 0
            break
    if flag:
        output_s = l

In [7]:
print("s' =", output_s)
print("s =", s)

s' = 1001
s = 1001
